# Prediction

In [1]:
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load trained artifacts
model = load_model("model.h5")

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

with open("config.pkl", "rb") as f:
    config = pickle.load(f)

max_len = config["max_len"]
print(f"Model loaded. max_len = {max_len}")

Model loaded. max_len = 14


In [2]:
def generate_response(text):
    seq = tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_len, padding='post')

    pred = model.predict(seq, verbose=0)
    pred = np.argmax(pred, axis=-1)[0]

    index_to_word = {v: k for k, v in tokenizer.word_index.items()}

    result = []
    for idx in pred:
        if idx != 0:
            word = index_to_word.get(idx, "")
            if word:
                result.append(word)

    return " ".join(result)

#### TEST 

In [3]:
test_cases = [
    "where is angkor wat specifically",
    "what food should i try in cambodia",
    "best time to visit cambodia",
    "is cambodia safe",
    "how to travel in phnom penh",
]

print("=" * 60)
for q in test_cases:
    print(f"Input : {q}")
    print(f"Output: {generate_response(q)}")
    print("-" * 60)

Input : where is angkor wat specifically
Output: it is is in in in in in in in in in in in
------------------------------------------------------------
Input : what food should i try in cambodia
Output: it should try amok and lok
------------------------------------------------------------
Input : best time to visit cambodia
Output: early morning at sunrise the the the the the the the the the the
------------------------------------------------------------
Input : is cambodia safe
Output: yes is generally generally generally generally generally generally generally generally generally generally generally generally
------------------------------------------------------------
Input : how to travel in phnom penh
Output: passapp or use tuk tuk tuks tuks tuks tuks tuks tuks tuks tuks tuks
------------------------------------------------------------


TEST2

In [4]:
error_cases = [
    ("Angkor Wat location?",              "Rephrased / short-form question"),
    ("WHERE IS ANGKOR WAT",               "Uppercase input — tokenizer is case-sensitive"),
    ("What's the weather like?",          "Out-of-domain question"),
    ("population of siem reap",           "Fact not in training data"),
    ("Can you book a hotel for me?",      "Action the model cannot perform"),
]

print("=" * 70)
for q, reason in error_cases:
    output = generate_response(q)
    print(f"Input          : {q}")
    print(f"Model output   : {output}")
    print(f"Failure reason : {reason}")
    print("-" * 70)

Input          : Angkor Wat location?
Model output   : siem it it it it it it it it it it it it it
Failure reason : Rephrased / short-form question
----------------------------------------------------------------------
Input          : WHERE IS ANGKOR WAT
Model output   : it is is in in in in in in in in in in in
Failure reason : Uppercase input — tokenizer is case-sensitive
----------------------------------------------------------------------
Input          : What's the weather like?
Model output   : to to to to to to to to to to to to to to
Failure reason : Out-of-domain question
----------------------------------------------------------------------
Input          : population of siem reap
Model output   : city north near near near near near near near near near near near near
Failure reason : Fact not in training data
----------------------------------------------------------------------
Input          : Can you book a hotel for me?
Model output   : it is usually safe safe safe safe

## Summary of Error Analysis

| Failure Type | Root Cause |
|---|---|
| Rephrased question | Model matches surface patterns, not meaning |
| Uppercase input | `Tokenizer` is case-sensitive by default |
| Out-of-domain question | No matching pattern in training data |
| Missing fact | Dataset does not contain that information |
| Impossible task | Model generates text only; cannot take actions |

**Core limitation:** SimpleRNN memorises input→output mappings. It has no semantic understanding, so any variation in wording causes it to fail. The vanishing-gradient problem also limits how well it handles long sentences.